# PRISM — Generation of 30 Reference Q&A Pairs from a Research Paper

This notebook generates up to **30 candidate question–answer (Q&A) pairs in English** from an original research paper (PDF) as the sole source document. Each pair includes a supporting evidence quote and its PDF page number. Generated Q&A pairs are **drafts requiring human verification** before use in scientific evaluation.

**Important:** This notebook uses the OpenAI API for Q&A generation and validation; it is separate from PRISM’s local LLM inference workflow. API usage may incur charges. The procedure does not optimize Q&A pairs for Ragas or any other evaluation score. Using the same LLM for both benchmark generation and evaluation may introduce bias. The notebook was designed for Google Colab.


In [ ]:
%pip -q install pymupdf pandas openai

## 1. Upload a PDF using the Colab Files sidebar

Upload the source PDF to `/content`. The following cell lists available PDFs and lets you select one by number.


In [ ]:
from pathlib import Path
PDF_FILES = sorted(Path("/content").glob("*.pdf"))
if not PDF_FILES:
    raise FileNotFoundError("No PDF found in /content. Upload a PDF using the Colab Files sidebar.")
for i, p in enumerate(PDF_FILES, 1):
    print(f"[{i}] {p.name}")
idx = int(input("Enter the number of the PDF to use: "))
if not 1 <= idx <= len(PDF_FILES):
    raise ValueError("Selection is out of range")
PDF_PATH = PDF_FILES[idx - 1]
print("Selected:", PDF_PATH.name)

## 2. Configure the model and API key

`TARGET_QA = 30` sets the target number of Q&A pairs. OpenAI API usage may incur charges.


In [ ]:
import os, getpass, json, re, time
import pandas as pd
import fitz
from openai import OpenAI

TARGET_QA = 30
MODEL = "gpt-5-mini" # Change if needed
MAX_PAGES = 100  # Safety limit for large PDFs
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
client = OpenAI()

## 3. Extract text from the PDF by page

The notebook attempts to exclude the references section. Verify text extraction order for two-column PDFs.


In [ ]:
pages=[]
with fitz.open(PDF_PATH) as doc:
    for page_i in range(min(len(doc), MAX_PAGES)):
        pg=doc[page_i]
        blocks=[x for x in pg.get_text("blocks") if len(x)>4 and isinstance(x[4],str) and x[4].strip()]
        # Approximate left-column to right-column order for two-column PDFs
        w=pg.rect.width
        blocks.sort(key=lambda x: (0 if (x[0]+x[2])/2 < w/2 else 1, x[1]))
        txt="\n".join(" ".join(x[4].split()) for x in blocks)
        if re.search(r"(?im)^references\s*$",txt):
            txt=re.split(r"(?im)^references\s*$",txt)[0]
            if txt.strip(): pages.append({"page":page_i+1,"text":txt})
            break
        if len(txt)>100: pages.append({"page":page_i+1,"text":txt})
print("Extracted pages:",len(pages))
print("Preview:",pages[0]["text"][:600] if pages else "EMPTY")
if not pages: raise ValueError("Could not extract text from the PDF")

## 4. Generate evidence-supported Q&A candidates

Each candidate includes `question`, `answer`, `evidence_quote`, and `page`. Evidence quotes must match text extracted from the PDF.


In [ ]:

import json
import time

def ask_json(system, user):
    for attempt in range(3):
        try:
            res = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ],
            )

            content = res.choices[0].message.content

            if not content:
                raise ValueError(
                    f"Empty response: {res.choices[0].finish_reason}"
                )

            return json.loads(content)

        except Exception as e:
            print(f"Attempt {attempt + 1}: {type(e).__name__}: {e}")

            if attempt == 2:
                raise

            time.sleep(2 * (attempt + 1))



SYSTEM = """
You are building an independently verifiable scientific
Q&A benchmark from a research paper.

Use ONLY the provided paper excerpts.
Generate English Q&A suitable for an interested non-specialist.

TARGET SCOPE:
- Study objectives and research design
- Experimental and clinical methods
- Main findings and clinical outcomes
- Safety and pathological findings
- Direct extraction of explicitly reported numerical values
- Conclusions and limitations

EXCLUDE:
- Bibliographic identification and cited references
- Questions about authors or publication years
- Counting or aggregation across multiple observations
- Calculations requiring multiple data points
- External knowledge and unsupported inference

REQUIREMENTS:
1. Every answer must be explicitly supported by the excerpt.
2. Questions must be clear, factual, and non-duplicative.
3. Answers must be concise and factually accurate.
4. Preserve numerical values and scientific terminology.
5. Provide a verbatim evidence quote for each answer.
6. Use the PDF page number supplied in the excerpt.
7. Do not optimize for any scoring system.

STRICT JSON OUTPUT:
Return exactly one JSON object with this structure:

{
  "items": [
    {
      "question": "Question text",
      "answer": "Answer text",
      "evidence_quote": "Exact supporting text",
      "page": 1
    }
  ]
}

The top-level key MUST be "items".
Every item MUST contain all four fields.
Do not use "Q&A" as a key.
Do not add commentary outside the JSON object.
"""


# Group extracted PDF text into two-page chunks
chunks = []

for i in range(0, len(pages), 2):
    group = pages[i:i+2]

    chunk = "\n\n".join(
        f"[PDF PAGE {p['page']}]\n{p['text'][:13000]}"
        for p in group
    )

    chunks.append(chunk)


# Check the output format using the first chunk
test_obj = ask_json(
    SYSTEM,
    f"Generate 3 factual Q&A items.\n\n{chunks[0]}"
)

print("TEST OUTPUT:")
print(json.dumps(test_obj, indent=2, ensure_ascii=False))


# Generate candidates from all chunks
candidates = []

for i, chunk in enumerate(chunks, 1):

    obj = ask_json(
        SYSTEM,
        f"""
Generate 15 diverse factual Q&A items from this excerpt.

Prioritize distinct scientific facts.
Avoid redundant questions.
Do not invent questions merely to reach 15 items.
If fewer than 15 valid questions are possible,
return only those that are supported.

{chunk}
"""
    )

    items = obj.get("items", [])

    accepted = 0

    for item in items:
        required = (
            "question",
            "answer",
            "evidence_quote",
            "page"
        )

        if isinstance(item, dict) and all(
            k in item for k in required
        ):
            candidates.append(item)
            accepted += 1

    print(
        f"Chunk {i}/{len(chunks)}: "
        f"{accepted} accepted, "
        f"{len(candidates)} total candidates"
    )

## 5. Validate evidence, remove duplicates, and select up to 30 Q&A pairs

Selection uses candidates grounded in the extracted paper text. If fewer than 30 valid candidates are available, the notebook reports the shortfall rather than fabricating additional pairs.


In [ ]:

import re
import json
from difflib import SequenceMatcher

# ==========================================
# 1. Verify evidence quotes against extracted text
# ==========================================

def norm(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()

verified = []
rejected = []

for x in candidates:
    q = str(x.get("question", "")).strip()
    a = str(x.get("answer", "")).strip()
    quote = " ".join(str(x.get("evidence_quote", "")).split())

    try:
        page = int(x.get("page"))
    except (ValueError, TypeError):
        rejected.append(("invalid_page", q))
        continue

    original = next(
        (p["text"] for p in pages if p["page"] == page),
        ""
    )

    if not (q and a and quote and norm(quote) in norm(original)):
        rejected.append(("invalid_evidence", q))
        continue

    verified.append({
        "Question": q,
        "Answer": a,
        "evidence_quote": quote,
        "evidence_page": page
    })

print("Candidates:", len(candidates))
print("Evidence verified:", len(verified))


# ==========================================
# 2. Use the LLM to assess relevance and semantic duplication
# ==========================================

VALIDATION_SYSTEM = """
You are validating a scientific Q&A benchmark.

Evaluate whether a candidate question is appropriate
and whether it duplicates an already accepted question.

IN_SCOPE:
- Study objectives and research design
- Experimental and clinical methods
- Clinical outcomes and main findings
- Safety and pathological findings
- Explicitly reported numerical values
- Conclusions and limitations

OUT_OF_SCOPE:
- Bibliographic identification
- Questions about cited external studies
- Calculations or aggregation across multiple observations
- Questions requiring external knowledge
- Ambiguous questions without a uniquely identifiable subject

DUPLICATION:
Two questions are duplicates when they ask for
essentially the same scientific fact, even if worded differently.

Questions about distinct patients, measurements,
methods, or outcomes are not automatically duplicates.

Also check whether the answer directly addresses
the question and is supported by the evidence quote.

Return ONLY a JSON object:
{
  "in_scope": true,
  "answer_supported": true,
  "duplicate": false
}
"""

valid = []

for i, x in enumerate(verified, 1):

    # Remove obvious text duplicates without API calls
    if any(
        SequenceMatcher(
            None,
            norm(x["Question"]),
            norm(y["Question"])
        ).ratio() > 0.90
        for y in valid
    ):
        rejected.append(("text_duplicate", x["Question"]))
        continue

    previous = [
        {
            "question": y["Question"],
            "answer": y["Answer"]
        }
        for y in valid
    ]

    prompt = json.dumps({
        "candidate": x,
        "previously_accepted": previous
    }, ensure_ascii=False)

    try:
        result = ask_json(VALIDATION_SYSTEM, prompt)

        in_scope = result.get("in_scope") is True
        supported = result.get("answer_supported") is True
        duplicate = result.get("duplicate") is True

        if not in_scope:
            rejected.append(("out_of_scope", x["Question"]))
            continue

        if not supported:
            rejected.append(("unsupported_answer", x["Question"]))
            continue

        if duplicate:
            rejected.append(("semantic_duplicate", x["Question"]))
            continue

        valid.append(x)

    except Exception as e:
        # Do not automatically accept candidates when API validation fails
        rejected.append(("validation_error", x["Question"]))
        print(f"Validation error at {i}: {e}")

    print(
        f"Checked {i}/{len(verified)} | "
        f"Accepted: {len(valid)}"
    )

print("\nVerified distinct:", len(valid))



# ==========================================
# 3. Select up to 30 questions
# ==========================================

TARGET_QA = 30

if len(valid) < TARGET_QA:
    selected = valid.copy()
    print(f"WARNING: {len(valid)}/{TARGET_QA} questions")

else:
    compact = [
        {
            "id": i,
            "question": x["Question"],
            "answer": x["Answer"],
            "page": x["evidence_page"]
        }
        for i, x in enumerate(valid)
    ]

    SELECT_SYSTEM = """
You are selecting a balanced scientific Q&A benchmark.

Select exactly the requested number of questions.

Prioritize:
- Study objectives and design
- Experimental and clinical methods
- Main findings
- Clinical outcomes
- Safety and pathological findings
- Conclusions and limitations

Avoid semantic duplication and overrepresentation
of the same scientific fact.

Return a valid JSON object containing an array named "ids".
Use only the integer IDs provided.
Do not invent IDs.
"""

    obj = ask_json(
        SELECT_SYSTEM,
        json.dumps({
            "target_count": TARGET_QA,
            "candidates": compact
        }, ensure_ascii=False)
    )

    ids = []

    for k in obj.get("ids", []):
        if (
            type(k) is int
            and 0 <= k < len(valid)
            and k not in ids
        ):
            ids.append(k)

    print("LLM selected:", len(ids))

    # If the LLM selects too few candidates,
    # fill the remaining slots using validated candidates
    if len(ids) < TARGET_QA:
        print("Additional validated candidates:", TARGET_QA - len(ids))

        for k in range(len(valid)):
            if len(ids) >= TARGET_QA:
                break

            if k not in ids:
                ids.append(k)

    selected = [
        valid[k] for k in ids[:TARGET_QA]
    ]

print("Selected:", len(selected), "/", TARGET_QA)



# ==========================================
# 4. Review results
# ==========================================

print("\n==============================")
print("Candidates:", len(candidates))
print("Evidence verified:", len(verified))
print("Verified distinct:", len(valid))
print("Selected:", len(selected), "/", TARGET_QA)
print("Rejected:", len(rejected))
print("==============================")

if len(selected) < TARGET_QA:
    print(
        "WARNING: Insufficient validated questions. "
        "Generate more candidates or review rejected items."
    )

# Summarize rejection reasons
from collections import Counter

print("\nRejection reasons:")
for reason, count in Counter(r for r, _ in rejected).items():
    print(f"{reason}: {count}")

# Display selected questions
for i, x in enumerate(selected, 1):
    print(f"{i}. {x['Question']}")


## 6. Export CSV files

`gold_qa_draft.csv` contains the `No,Question,Answer` columns. `gold_qa_evidence_audit.csv` includes evidence and review status for manual verification.


In [ ]:
out=pd.DataFrame(selected)
out.insert(0,"No",range(1,len(out)+1))
out["review_status"]="Draft - human verification required"
out[["No","Question","Answer"]].to_csv("/content/gold_qa_draft.csv",index=False,encoding="utf-8-sig")
out.to_csv("/content/gold_qa_evidence_audit.csv",index=False,encoding="utf-8-sig")
print("Saved: /content/gold_qa_draft.csv")
print("Saved: /content/gold_qa_evidence_audit.csv")
display(out.head(10))

## 7. Download results in Google Colab


In [ ]:
from google.colab import files
files.download("/content/gold_qa_draft.csv")
files.download("/content/gold_qa_evidence_audit.csv")